# 05. 장르 × 성과 등급 분석

**분석 목적:** 성과 등급(규모 × 만족도)을 장르와 교차 분석하여, 인디 개발사가 장르를 선택할 때 참고할 수 있는 데이터 기반 근거를 제공한다.

**사용 데이터:** `data/preprocessed/steam_indie_games_graded.csv` (9,692개, 성과 등급 컬럼 포함)

**분석 흐름:**
1. 데이터 로드 및 장르 explode
2. 장르별 성과 등급 분포 히트맵
3. 장르별 규모 vs 만족도 포지셔닝 (버블 차트)
4. 장르별 흥행 전환율
5. 출시 연도 × 장르 성과 추이

## 0. 라이브러리 로드 및 공통 설정

In [8]:
import ast
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']

PALETTE = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B2', '#937860', '#DA8BC3', '#8C8C8C'
]
COLOR_MAP = {g: PALETTE[i] for i, g in enumerate(TARGET_GENRES)}

GRADE_ORDER = [
    'high_high', 'high_mid', 'high_low',
    'mid_high',  'mid_mid',  'mid_low',
    'low_high',  'low_mid',  'low_low',
]
GRADE_LABEL = {
    'high_high': '대흥행',
    'high_mid' : '상업적 성공',
    'high_low' : '호불호',
    'mid_high' : '숨겨진 명작',
    'mid_mid'  : '평범',
    'mid_low'  : '외면',
    'low_high' : '니치 (틈새)',
    'low_mid'  : '미노출',
    'low_low'  : '미반응',
}
GRADE_LABEL_ORDER = [GRADE_LABEL[g] for g in GRADE_ORDER]

## 1. 데이터 로드 및 장르 explode

In [9]:
games = pd.read_csv('../../../data/preprocessed/steam_indie_games_graded.csv')
games['release_year'] = pd.to_datetime(games['release_date']).dt.year
games['genres_list'] = games['genres'].apply(lambda g: ast.literal_eval(g) if pd.notna(g) else [])
games['genres_filtered'] = games['genres_list'].apply(
    lambda gl: [g for g in gl if g in TARGET_GENRES]
)

games_with_genre = games[games['genres_filtered'].map(len) > 0].copy()
df = (
    games_with_genre
    .explode('genres_filtered')
    .rename(columns={'genres_filtered': 'genre'})
    .reset_index(drop=True)
)
df['grade_label'] = df['performance_grade'].map(GRADE_LABEL)

print(f'원본 게임 수    : {len(games_with_genre):,}개')
print(f'explode 후 행 수: {len(df):,}행 (중복 포함)')
print()
print('장르별 집계 게임 수:')
print(df['genre'].value_counts().to_string())

원본 게임 수    : 9,169개
explode 후 행 수: 19,955행 (중복 포함)

장르별 집계 게임 수:
genre
Adventure     4732
Casual        4051
Action        3995
Simulation    2429
RPG           2155
Strategy      1980
Sports         329
Racing         284


## 2. 장르별 성과 등급 분포 히트맵

각 장르 내에서 성과 등급이 차지하는 비율을 히트맵으로 시각화한다.
비율 기준으로 표시하여 게임 수가 다른 장르 간 공정한 비교가 가능하다.

In [10]:
# 장르 × 등급 교차표 (비율)
cross = pd.crosstab(df['genre'], df['grade_label'], normalize='index') * 100
cross = cross.reindex(columns=GRADE_LABEL_ORDER)

# 대흥행 비율 기준으로 장르 정렬
cross = cross.sort_values('대흥행', ascending=False)

fig = px.imshow(
    cross.round(1),
    text_auto='.1f',
    color_continuous_scale='Blues',
    aspect='auto',
    title='장르별 성과 등급 비율 히트맵 (%, 행 합계 = 100)<br><sub>다중 장르 중복 집계 / 대흥행 비율 내림차순 정렬</sub>',
    labels={'x': '성과 등급', 'y': '장르', 'color': '비율 (%)'},
)
fig.update_layout(height=420, coloraxis_colorbar_title='%')
fig.show()

**해석:**
- **대흥행(high_high) 비율이 높은 장르:** 해당 속성을 가진 게임이 리뷰 500개 이상 + 긍정률 80% 이상을 동시에 달성하는 빈도가 높다.
- **니치(low_high) 비율이 높은 장르:** 유저 만족도는 높지만 노출·규모가 작은 틈새 시장 성격이 강하다.
- **미반응(low_low) 비율이 높은 장르:** 진입 대비 성과가 낮은 구간으로, 해당 장르에서 차별화 전략이 더욱 중요하다.

## 3. 장르별 규모 vs 만족도 포지셔닝 (버블 차트)

x축: 중앙값 리뷰 수 (규모), y축: 중앙값 긍정률 (만족도), 버블 크기: 게임 수
4분면으로 장르의 시장 포지션을 직관적으로 파악한다.

In [11]:
genre_agg = df.groupby('genre').agg(
    리뷰수_중앙값=('total_reviews', 'median'),
    긍정률_중앙값=('positive_rate', 'median'),
    게임수=('appid', 'count'),
    대흥행_비율=('performance_grade', lambda x: (x == 'high_high').mean() * 100),
).reset_index().round(2)

review_mid = genre_agg['리뷰수_중앙값'].median()
pos_mid = genre_agg['긍정률_중앙값'].median()

fig = px.scatter(
    genre_agg,
    x='리뷰수_중앙값',
    y='긍정률_중앙값',
    size='게임수',
    color='genre',
    color_discrete_map=COLOR_MAP,
    text='genre',
    size_max=60,
    hover_data={'대흥행_비율': ':.1f', '게임수': True},
    title='장르별 포지셔닝: 시장 규모 × 유저 만족도<br><sub>버블 크기 = 게임 수 / 중앙값 기준 / 다중 장르 중복 집계</sub>',
    labels={'리뷰수_중앙값': '중앙값 리뷰 수 (규모)', '긍정률_중앙값': '중앙값 긍정률 (%)'},
)
fig.update_traces(textposition='top center', marker=dict(opacity=0.8))

fig.add_vline(x=review_mid, line_dash='dot', line_color='gray', opacity=0.6,
              annotation_text='규모 중앙값', annotation_position='top right')
fig.add_hline(y=pos_mid, line_dash='dot', line_color='gray', opacity=0.6,
              annotation_text='만족도 중앙값', annotation_position='top right')

fig.update_layout(showlegend=False, height=520,
                  yaxis=dict(range=[80, 93]))
fig.show()

print('\n장르별 집계 요약:')
display(genre_agg.set_index('genre').sort_values('리뷰수_중앙값', ascending=False))


장르별 집계 요약:


,리뷰수_중앙값,긍정률_중앙값,게임수,대흥행_비율
genre,,,,
RPG,70.0,85.76,2155,12.99
Simulation,56.0,83.87,2429,12.68
Strategy,52.0,86.36,1980,11.57
Adventure,44.0,87.50,4732,9.59
Action,38.0,87.88,3995,9.49
Sports,37.0,85.71,329,4.86
Casual,35.0,89.79,4051,7.75
Racing,27.5,87.60,284,6.69


**4분면 해석:**

| 위치 | 특성 | 전략 제안 |
|------|------|----------|
| 우상단 (규모↑ 만족도↑) | 주류 시장, 높은 경쟁 | 품질 차별화 필수, 성공 시 파급력 큼 |
| 좌상단 (규모↓ 만족도↑) | 틈새 시장, 팬층 두터움 | 커뮤니티 마케팅으로 규모 확장 가능 |
| 우하단 (규모↑ 만족도↓) | 기대치 대비 실망 큼 | 출시 전 QA·완성도 집중 |
| 좌하단 (규모↓ 만족도↓) | 시장·만족도 모두 약함 | 신중한 장르 선택 또는 세부 차별화 필요 |

## 4. 장르별 흥행 전환율

각 장르에서 만족도 높은 구간(대흥행 + 숨겨진 명작 + 니치)과
미반응 구간(미반응 + 외면 + 미노출)의 비율을 비교한다.

In [12]:
# 만족도 기준 구간 분류
HIGH_SAT = ['high_high', 'mid_high', 'low_high']   # 만족도 높음 (긍정률 80% 이상)
MID_SAT  = ['high_mid',  'mid_mid',  'low_mid']    # 만족도 중간 (70~79%)
LOW_SAT  = ['high_low',  'mid_low',  'low_low']    # 만족도 낮음 (70% 미만)

def classify_sat(grade):
    if grade in HIGH_SAT:
        return '높은 만족도 (≥80%)'
    elif grade in MID_SAT:
        return '보통 만족도 (70~79%)'
    else:
        return '낮은 만족도 (<70%)'

df['sat_group'] = df['performance_grade'].apply(classify_sat)
SAT_ORDER = ['높은 만족도 (≥80%)', '보통 만족도 (70~79%)', '낮은 만족도 (<70%)']
SAT_COLORS = {'높은 만족도 (≥80%)': '#2d6a4f', '보통 만족도 (70~79%)': '#adb5bd', '낮은 만족도 (<70%)': '#C44E52'}

sat_pct = (
    df.groupby(['genre', 'sat_group'])
    .size()
    .reset_index(name='count')
)
sat_pct['pct'] = sat_pct.groupby('genre')['count'].transform(lambda x: x / x.sum() * 100)

# 높은 만족도 비율 기준으로 장르 정렬
genre_high_sat_order = (
    sat_pct[sat_pct['sat_group'] == '높은 만족도 (≥80%)']
    .set_index('genre')['pct']
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.bar(
    sat_pct,
    x='genre',
    y='pct',
    color='sat_group',
    color_discrete_map=SAT_COLORS,
    category_orders={'genre': genre_high_sat_order, 'sat_group': SAT_ORDER},
    text='pct',
    barmode='stack',
    title='장르별 만족도 구간 비율 (스택 바)<br><sub>다중 장르 중복 집계</sub>',
    labels={'genre': '장르', 'pct': '비율 (%)', 'sat_group': '만족도 구간'},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside', textfont_size=11)
fig.update_layout(height=450, yaxis_range=[0, 105])
fig.show()

In [13]:
# 규모 기준 구간 분류
HIGH_SCALE = ['high_high', 'high_mid', 'high_low']  # 규모 높음 (리뷰 500개 이상)
MID_SCALE  = ['mid_high',  'mid_mid',  'mid_low']   # 규모 중간
LOW_SCALE  = ['low_high',  'low_mid',  'low_low']   # 규모 낮음

conversion = (
    df.groupby('genre')['performance_grade']
    .agg(
        대흥행_비율=lambda x: (x == 'high_high').mean() * 100,
        만족도높음_비율=lambda x: x.isin(HIGH_SAT).mean() * 100,
        규모높음_비율=lambda x: x.isin(HIGH_SCALE).mean() * 100,
        미반응_비율=lambda x: (x == 'low_low').mean() * 100,
    )
    .round(1)
    .sort_values('대흥행_비율', ascending=False)
)
conversion.columns = ['대흥행_비율(%)', '만족도높음_비율(%)', '규모높음_비율(%)', '미반응_비율(%)']

print('장르별 흥행 지표 요약:')
display(conversion)

장르별 흥행 지표 요약:


,대흥행_비율(%),만족도높음_비율(%),규모높음_비율(%),미반응_비율(%)
genre,,,,
RPG,13.0,66.1,18.0,9.0
Simulation,12.7,59.7,16.8,13.2
Strategy,11.6,66.8,16.5,8.4
Adventure,9.6,68.7,12.7,10.4
Action,9.5,69.0,12.8,10.0
Casual,7.8,73.6,9.7,9.4
Racing,6.7,65.8,8.1,13.0
Sports,4.9,66.9,6.4,12.2


**해석:**
- **만족도 높음 비율:** 해당 장르 속성을 가진 게임 중 긍정률 80% 이상을 달성하는 비율 — 품질 달성 가능성을 나타낸다.
- **대흥행 비율:** 리뷰 500개 이상 + 긍정률 80% 이상을 동시에 달성하는 비율 — 장르의 진정한 흥행 전환율이다.
- **미반응 비율:** 리뷰 49개 이하 + 긍정률 70% 미만인 비율 — 장르 진입 리스크를 나타낸다.

인디 개발사라면 **대흥행 비율이 높고 미반응 비율이 낮은 장르**를 우선 고려할 수 있다.

## 5. 출시 연도 × 장르 성과 추이

2023~2025년 기간 동안 장르별 대흥행 비율과 미반응 비율이 어떻게 변화했는지 확인한다.

In [14]:
trend = (
    df.groupby(['release_year', 'genre'])['performance_grade']
    .agg(
        대흥행_비율=lambda x: (x == 'high_high').mean() * 100,
        미반응_비율=lambda x: (x == 'low_low').mean() * 100,
        게임수=lambda x: len(x),
    )
    .round(2)
    .reset_index()
)

# 연도별 충분한 데이터가 있는 장르만 (각 연도 20개 이상)
trend = trend[trend['게임수'] >= 20]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('장르별 대흥행 비율 추이', '장르별 미반응 비율 추이'),
    shared_xaxes=True,
)

for genre in TARGET_GENRES:
    g = trend[trend['genre'] == genre]
    if len(g) < 2:
        continue
    color = COLOR_MAP[genre]
    fig.add_trace(go.Scatter(
        x=g['release_year'], y=g['대흥행_비율'],
        mode='lines+markers', name=genre,
        line=dict(color=color), marker=dict(size=7),
        legendgroup=genre,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=g['release_year'], y=g['미반응_비율'],
        mode='lines+markers', name=genre,
        line=dict(color=color, dash='dot'), marker=dict(size=7),
        legendgroup=genre, showlegend=False,
    ), row=1, col=2)

fig.update_layout(
    title='출시 연도 × 장르 성과 추이 (2023~2025)<br><sub>대흥행: 리뷰 500개↑ + 긍정률 80%↑ / 미반응: 리뷰 49개↓ + 긍정률 70%↓</sub>',
    height=480,
    legend=dict(title='장르'),
)
fig.update_xaxes(tickvals=[2023, 2024, 2025], ticktext=['2023', '2024', '2025'])
fig.update_yaxes(title_text='비율 (%)', row=1, col=1)
fig.update_yaxes(title_text='비율 (%)', row=1, col=2)
fig.show()

In [15]:
trend_pivot = trend.pivot_table(
    index='genre', columns='release_year',
    values='대흥행_비율', aggfunc='first'
).round(1)
trend_pivot.columns.name = '출시연도 (대흥행 비율 %)'

print('연도별 대흥행 비율 (%):')
display(trend_pivot.sort_values(2025, ascending=False, na_position='last'))

연도별 대흥행 비율 (%):


출시연도 (대흥행 비율 %),2023.0,2024.0,2025.0
genre,,,
Simulation,11.5,12.9,13.3
RPG,13.4,13.1,12.3
Strategy,10.8,12.2,11.1
Racing,4.7,5.7,10.6
Action,9.3,9.9,8.7
Adventure,10.4,10.1,7.8
Casual,8.4,7.7,6.6
Sports,3.8,4.8,3.4


**해석:**
- 대흥행 비율이 **증가하는 장르**는 시장이 성장 중이거나 작품 수 대비 흥행작 비율이 높아지는 추세다.
- 대흥행 비율이 **감소하고 미반응 비율이 증가하는 장르**는 공급 과잉 또는 시장 포화 신호일 수 있다.
- 단, 2025년은 데이터 수집 시점 기준으로 출시 후 충분한 시간이 경과하지 않은 게임이 포함될 수 있어 비율이 낮게 나타날 수 있다.

## 6. 종합 요약

### 인디 개발사를 위한 장르 선택 인사이트

| 분석 | 핵심 인사이트 |
|------|-------------|
| 등급 분포 히트맵 | 장르별 성과 등급 비율 → 어떤 장르에서 대흥행/틈새/미반응 비율이 높은지 파악 |
| 포지셔닝 버블 차트 | 규모 vs 만족도 → 경쟁 강도와 팬층 특성의 장르별 차이 확인 |
| 흥행 전환율 | 장르별 대흥행·미반응 비율 → 진입 기대값과 리스크 수치화 |
| 연도별 추이 | 장르별 시장 성장·포화 방향 → 타이밍 전략 판단 근거 |

### 활용 가이드

| 상황 | 추천 장르 선택 기준 |
|------|-------------------|
| 첫 출시, 리소스 제한 | 만족도 높음 비율↑ + 미반응 비율↓ 장르 → 실패 리스크 최소화 |
| 시장 규모 우선 | 규모 높음 비율↑ 장르 → 단, 품질 차별화 필수 |
| 틈새 공략 | 버블 차트 좌상단 장르 → 커뮤니티 마케팅 병행 |
| 트렌드 타기 | 최근 2년 대흥행 비율 상승 장르 → 출시 타이밍 전략과 연계 |